In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
PYTORCH_ENABLE_MPS_FALLBACK=1 

# =========================================================
# 1. Data loading
# =========================================================
def load_file(filepath):
    return pd.read_csv(filepath, header=None, sep=r"\s+").values


def load_group(filenames, prefix):
    loaded = [load_file(prefix / name) for name in filenames]
    return np.dstack(loaded)  # shape: (samples, timesteps, features)


def load_dataset_group(group, prefix):
    filepath = prefix / group / "Inertial Signals"

    filenames = [
        f"total_acc_x_{group}.txt", f"total_acc_y_{group}.txt", f"total_acc_z_{group}.txt",
        f"body_acc_x_{group}.txt",  f"body_acc_y_{group}.txt",  f"body_acc_z_{group}.txt",
        f"body_gyro_x_{group}.txt", f"body_gyro_y_{group}.txt", f"body_gyro_z_{group}.txt",
    ]

    X = load_group(filenames, filepath)
    y = load_file(prefix / group / f"y_{group}.txt").squeeze().astype(int) - 1
    return X, y


def load_dataset(prefix="UCI HAR Dataset"):
    prefix = Path(prefix)

    trainX, trainy = load_dataset_group("train", prefix)
    testX, testy = load_dataset_group("test", prefix)

    print("Loaded shapes:")
    print("trainX:", trainX.shape, "trainy:", trainy.shape)
    print("testX: ", testX.shape, "testy: ", testy.shape)

    return trainX, trainy, testX, testy


# =========================================================
# 2. Dataset
# =========================================================
class HARDataset(Dataset):
    """
    Input:
        X shape = (samples, timesteps, features)

    Convert to PyTorch Conv1d format:
        (samples, features, timesteps)
    """
    def __init__(self, X, y):
        X = np.transpose(X, (0, 2, 1))  # -> (samples, features, timesteps)
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# =========================================================
# 3. One sensor-processing head
# =========================================================
class SensorHead(nn.Module):
    def __init__(self, in_channels=3, filters=64, kernel_size=5, dropout=0.5):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_channels=in_channels, out_channels=filters, kernel_size=kernel_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.MaxPool1d(kernel_size=2),
        )

    def forward(self, x):
        return self.block(x)


# =========================================================
# 4. Multi-head model by sensor type
# =========================================================
class SensorTypeCNN(nn.Module):
    """
    Channel layout in UCI HAR:
      0:3 -> total_acc_x/y/z
      3:6 -> body_acc_x/y/z
      6:9 -> body_gyro_x/y/z
    """
    def __init__(self, n_timesteps, n_outputs, filters=64, kernel_size=5, dropout=0.5):
        super().__init__()

        self.total_acc_head = SensorHead(
            in_channels=3, filters=filters, kernel_size=kernel_size, dropout=dropout
        )
        self.body_acc_head = SensorHead(
            in_channels=3, filters=filters, kernel_size=kernel_size, dropout=dropout
        )
        self.body_gyro_head = SensorHead(
            in_channels=3, filters=filters, kernel_size=kernel_size, dropout=dropout
        )

        # Infer flattened dimension automatically
        with torch.no_grad():
            dummy = torch.zeros(1, 9, n_timesteps)

            total_acc = dummy[:, 0:3, :]
            body_acc = dummy[:, 3:6, :]
            body_gyro = dummy[:, 6:9, :]

            h1 = self.total_acc_head(total_acc).reshape(1, -1)
            h2 = self.body_acc_head(body_acc).reshape(1, -1)
            h3 = self.body_gyro_head(body_gyro).reshape(1, -1)

            merged_dim = h1.shape[1] + h2.shape[1] + h3.shape[1]

        self.classifier = nn.Sequential(
            nn.Linear(merged_dim, 100),
            nn.ReLU(),
            nn.Linear(100, n_outputs),
        )

    def forward(self, x):
        # Split by sensor type
        total_acc = x[:, 0:3, :]   # total/device acceleration
        body_acc = x[:, 3:6, :]    # body acceleration
        body_gyro = x[:, 6:9, :]   # body gyroscope

        h1 = self.total_acc_head(total_acc).reshape(x.size(0), -1)
        h2 = self.body_acc_head(body_acc).reshape(x.size(0), -1)
        h3 = self.body_gyro_head(body_gyro).reshape(x.size(0), -1)

        merged = torch.cat([h1, h2, h3], dim=1)
        out = self.classifier(merged)
        return out


# =========================================================
# 5. Train / evaluate
# =========================================================
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_n = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        total_correct += (logits.argmax(dim=1) == yb).sum().item()
        total_n += xb.size(0)

    return total_loss / total_n, total_correct / total_n


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_n = 0

    for xb, yb in loader:
        xb = xb.to(device)
        yb = yb.to(device)

        logits = model(xb)
        loss = criterion(logits, yb)

        total_loss += loss.item() * xb.size(0)
        total_correct += (logits.argmax(dim=1) == yb).sum().item()
        total_n += xb.size(0)

    return total_loss / total_n, total_correct / total_n


# =========================================================
# 6. Main fit/evaluate function
# =========================================================
def evaluate_model(trainX, trainy, testX, testy, epochs=10, batch_size=32, lr=1e-3):
    device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')

    n_timesteps = trainX.shape[1]
    n_outputs = len(np.unique(trainy))

    train_dataset = HARDataset(trainX, trainy)
    test_dataset = HARDataset(testX, testy)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

    model = SensorTypeCNN(
        n_timesteps=n_timesteps,
        n_outputs=n_outputs,
        filters=64,
        kernel_size=5,
        dropout=0.5,
    ).to(device)

    try:
        model = torch.compile(model)
        print("Using torch.compile()")
    except Exception:
        print("torch.compile() unavailable, continuing without it")

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        test_loss, test_acc = evaluate(model, test_loader, criterion, device)

        print(
            f"Epoch {epoch+1:02d}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
            f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}"
        )

    return model, test_acc




In [3]:
# =========================================================
# 7. Run directly
# =========================================================

trainX, trainy, testX, testy = load_dataset("data/UCI_HAR_Dataset")
model, test_acc = evaluate_model(
    trainX, trainy, testX, testy,
    epochs=20,
    batch_size=32,
    lr=1e-3
)
print(f"\nFinal test accuracy: {test_acc:.4f}")

Loaded shapes:
trainX: (7352, 128, 9) trainy: (7352,)
testX:  (2947, 128, 9) testy:  (2947,)
Using torch.compile()
Epoch 01/20 | Train Loss: 0.4008 | Train Acc: 0.8366 | Test Loss: 0.3893 | Test Acc: 0.8314
Epoch 02/20 | Train Loss: 0.1637 | Train Acc: 0.9348 | Test Loss: 0.2818 | Test Acc: 0.8982
Epoch 03/20 | Train Loss: 0.1260 | Train Acc: 0.9471 | Test Loss: 0.3003 | Test Acc: 0.8826
Epoch 04/20 | Train Loss: 0.1239 | Train Acc: 0.9505 | Test Loss: 0.2958 | Test Acc: 0.8992
Epoch 05/20 | Train Loss: 0.1041 | Train Acc: 0.9538 | Test Loss: 0.3079 | Test Acc: 0.9009
Epoch 06/20 | Train Loss: 0.0977 | Train Acc: 0.9584 | Test Loss: 0.2932 | Test Acc: 0.9009
Epoch 07/20 | Train Loss: 0.0967 | Train Acc: 0.9576 | Test Loss: 0.3115 | Test Acc: 0.9050
Epoch 08/20 | Train Loss: 0.0894 | Train Acc: 0.9596 | Test Loss: 0.3317 | Test Acc: 0.9101
Epoch 09/20 | Train Loss: 0.0906 | Train Acc: 0.9596 | Test Loss: 0.3964 | Test Acc: 0.8992
Epoch 10/20 | Train Loss: 0.0882 | Train Acc: 0.9626 | Te